In [1]:
# Jaxlib
import jax
from jax import lax
from jax import random as jrnd
from jax import numpy as jnp
from jax import tree_util as jtu
jax.config.update('jax_enable_x64', True)

# Others
from matplotlib import pyplot as plt

# This
from numerics import *
from seismic import *

In [2]:
n = 20
d = 5
x = jrnd.normal(jrnd.key(0), (n, d)) * jrnd.uniform(jrnd.key(1), (n, 1), minval = 0.5, maxval = 5)
x = jnp.sort(x, axis = 0)

mu = x.mean(axis = -1)
sigma = x.std(axis = -1)

crossvar = jnp.outer(sigma, sigma)
covarg = jnpla.norm(x[None] - x[:, None], axis = -1) / d
lscale = 1 / sigma
kern = crossvar * ((2 * lscale[None] * lscale[:, None]) / (lscale[None]**2 + lscale[:, None]**2)) ** (1 / 2) * jnp.exp(-covarg**2 / (lscale[None]**2 + lscale[:, None]**2))

In [3]:
site = Site(0., 0., 760., 0.5, 2.9, 1.)
mfd1 = MFD(4., .8, 4., 8.)
mfd2 = MFD(4., 1., 4., 8.)
mfd3 = MFD(5., 1.2, 4., 5.)
fault1 = Fault(0., 20., 3.0, 0.1, 40., 80., 90., 2.5, 0., mfd1)
fault2 = Fault(0., 25., 0.5, 0.2, 80., 80., 30., 0.5, 1., mfd2)
fault3 = Fault(22., 12.1, 3.9, 1.2, 130., 40., 22., 3., 1., mfd3)
faults = [fault1, fault2, fault3]
fault_tree = make_fault_tree(faults)
fault_tree1 = make_fault_tree([faults[0]])
fault_tree2 = make_fault_tree([faults[1]])

scn = Scenario(site, fault_tree)
scn1 = Scenario(site, fault_tree1)
scn2 = Scenario(site, fault_tree2)

In [4]:
# VARIANCE CALCULATION: Pretty straightforward, V = (w * (lnSA - mu_lnSA)**2).sum()

n = 8
m = 4
t = jnp.linspace(-jnp.pi, jnp.pi, n)[:, None]
x_all = jnp.sin(t) + jrnd.normal(jrnd.key(0),shape = (n, m)) * 0.25
w = jrnd.uniform(jrnd.key(1), (m,))
w = w / w.sum()

# Mean and variance.
mu = (x_all @ w)
sig2 = (x_all - mu[:, None])**2 @ w
Sig2 = jnp.diag(sig2)

# Woohoo. There's our covariance matrix.
Cov = jax.vmap(jnp.outer, in_axes = (1, 1), out_axes = 2)(x_all - mu[:, None], x_all - mu[:, None])
Cov = jnp.einsum('ijk,k->ij', Cov, w)

In [5]:
@jtu.register_pytree_node_class
class GMMLT:
    def __init__(self, gmms:list, T:float, weights:jax.typing.ArrayLike):
        self.T = T
        self.gmms = gmms
        self.weights = weights

    # Calculate for a single GMM. Takes R so we don't repeat the calculation every time.
    def calc_single(self, i:int, Mw:float, site:Site, fault:Fault, R:jax.Array):
        return lax.switch(i, self.gmms, Mw, self.T, site, fault, R)

    def tree_flatten(self):
        return (self.T, self.weights), self.gmms
    
    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(aux, *children)

def calc_haz(x:jax.Array, gmmlt:GMMLT, scn:Scenario, dM:float = 0.1):
    M_min, M_max = scn.fault_tree.mfd.M_min, scn.fault_tree.mfd.M_max

    # Magnitude bins.
    bins_M = jnp.arange(M_min.min(), M_max.max() + dM, dM)
    # MFD rates
    n_M = jax.vmap(scn.fault_tree.mfd.calc_lmdaM)(bins_M)
    # Take difference to convert from exceedance rates to discrete binned rates
    n_M_inc = n_M[:-1] - n_M[1:]

    # Roots for GMM evaluation.
    roots_M = (bins_M[:-1] + bins_M[1:]) / 2
    # Array of ones/zeros for each fault signifying array inside/outside range (shape (roots_M.shape, fault_num))
    weights_mask = (roots_M[:, None] > M_min[None, :]) & (roots_M[:, None] < M_max[None, :])
    # We can think of our incremental rates as already including quadrature weights, so 
    #   we'll just multiply them by the mask to be safe. 
    n_M_inc = n_M_inc * weights_mask

    # Ground motion means + stds for each fault at roots
    # Calculate R for full fault tree...
    R_tree = jax.vmap(calc_R, in_axes = (None, 0))(scn.site, scn.fault_tree)
    # Triple vmap. First, across faults (and corresponding distances),
    calc_faults = jax.vmap(gmmlt.calc_single, in_axes=(None, None, None, 0, 0))
    # Then across magnitudes,
    calc_M = jax.vmap(calc_faults, in_axes=(None, 0, None, None, None))
    # Then across GMMs. This order minimizes recompilation.
    calc_gmms = jax.vmap(calc_M, in_axes=(0, None, None, None, None))

    # Grab indices and vmap across
    gmm_idcs = jnp.arange(len(gmmlt.gmms))
    all_mu_lnSA, all_std_lnSA = calc_gmms(gmm_idcs, roots_M, scn.site, scn.fault_tree, R_tree)
    # Get PoE at all points
    all_prob_x = 1 - jax.vmap(trunc_norm_cdf, in_axes = (0, None, None, None, None))(jnp.log(x), 
                                    -jnp.inf, 3 * all_std_lnSA, 
                                    all_mu_lnSA, all_std_lnSA)

    # Take mean
    mu_prob_x = jnp.einsum('i,hijk->hjk', gmmlt.weights, all_prob_x)

    # Hazard integrand (magnitude probabilities * exceedance probabilities)
    haz_intgrnd = mu_prob_x * n_M_inc

    # Return the sum since our "quadrature weights" are basically
    #   included in our frequency bins
    return jnp.einsum('ijk->i',haz_intgrnd), 

gmms = [f_ASK14, f_BSSA14]
gmmlt = GMMLT(gmms, 0.3, jnp.array([0.5, 0.5]))

x = 0.2
xv = jnp.linspace(0.01, 0.5, 50)
haz = calc_haz(xv, gmmlt, scn)
# Monte carlo for GMMs alone is pretty simple:
# Get hazard estimate for each individual GMM, take weighted sample, calculate StD
# Including probabilistic ERF it becomes more difficult. But this is just a toy example for now.